# weight-decay-l2-add — ex2: run 2 Adam-steps with L2-into-grad and AdamW side by side; verify divergence

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `weight-decay-l2-add`. Running the final beacon cell reports progress against the `Optimizer: Weight decay L2` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Weight decay L2` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`weight-decay-l2-add`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "weight-decay-l2-add"
DD_SUBTOPIC = "Optimizer: Weight decay L2"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## L2-into-gradient over 2 steps — AdamW divergence visible quickly

Ex1 folded `λ·θ` into `g` — the canonical L2 regularisation move. This is fine for plain SGD (it's exactly equivalent to penalising ½λ‖θ‖² in the loss). The deepening move shows where it BREAKS: stacked with Adam's adaptive normalisation, the L2 path and the decoupled (AdamW) path diverge after just 2 steps on a single scalar parameter.

**The experiment.** Same θ₀, same g₁, same g₂, same hparams. Two trajectories:
- **L2 path:** at each step, compute `g' = g + λ·θ`, run Adam on `g'`.
- **AdamW path:** at each step, run Adam on `g`, then `θ ← θ − lr·λ·θ`.

After 2 steps, `θ_L2 ≠ θ_AdamW`. The gap grows with `v̂`'s asymmetry across steps — small-gradient steps amplify the decoupling difference.

**Why this matters for ex1.** `apply_weight_decay` is the right tool for SGD-style optimisers; chaining it INTO Adam's gradient stream silently couples decay to the adaptive scale. The fix isn't to fix L2 — it's to switch to AdamW.

### Exercise 2 — run 2 Adam-steps with L2-into-grad and AdamW side by side; verify divergence

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyse where L2-into-gradient (ex1's `apply_weight_decay`) breaks when chained with Adam: simulate two Adam steps along the L2 path and along the decoupled (AdamW) path and verify the resulting θ trajectories diverge.
> Keywords: l2, adam, adamw, two-step
> ```

**KCs targeted:** `l2-fold-into-gradient`, `two-step-state-accumulation`

Implement `ex2_two_step_l2_vs_adamw(theta0, grads, lr, beta1, beta2, eps, lmda)`.

`grads` is a `list[Tensor]` of length 2 — the gradients at step 1 and step 2 (assumed already computed by the caller — no autograd needed).

Run TWO trajectories of TWO steps each, both starting from the same `theta0` with `m = v = 0`. Steps are 1-indexed.

**L2 path** (ex1's `apply_weight_decay` folded into Adam's grad):
```
for step in (1, 2):
    g = grads[step-1] + lmda * theta
    m ← beta1·m + (1-beta1)·g
    v ← beta2·v + (1-beta2)·g²
    m̂ = m / (1 - beta1**step)
    v̂ = v / (1 - beta2**step)
    theta ← theta - lr · (m̂ / (√v̂ + eps))
```

**AdamW path** (decoupled):
```
for step in (1, 2):
    g = grads[step-1]                    # no fold
    m ← beta1·m + (1-beta1)·g
    v ← beta2·v + (1-beta2)·g²
    m̂ = m / (1 - beta1**step)
    v̂ = v / (1 - beta2**step)
    theta ← theta - lr · (m̂ / (√v̂ + eps)) - lr · lmda · theta
```

Return tuple `(theta_l2_final, theta_adamw_final)` after both trajectories complete 2 steps.

In [ ]:
def ex2_two_step_l2_vs_adamw(theta0, grads, lr, beta1, beta2, eps, lmda):
    """Run 2 steps of L2-path-Adam and AdamW from the same θ₀; return (θ_l2, θ_adamw)."""
    raise NotImplementedError()


def _test_ex2():
    # === Two trajectories complete, return fresh tensors ===
    theta0 = t.tensor([1.0, -1.0])
    grads = [t.tensor([0.1, -0.05]), t.tensor([0.08, -0.03])]
    th_l2, th_aw = ex2_two_step_l2_vs_adamw(theta0, grads, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, lmda=0.1)
    assert th_l2.shape == theta0.shape
    assert th_aw.shape == theta0.shape

    # === The two trajectories diverge after 2 steps (the WHOLE POINT) ===
    diff = (th_l2 - th_aw).abs()
    assert diff.max() > 1e-6, (
        f'L2-into-Adam and AdamW must yield different θ after 2 steps; '
        f'got identical: {th_l2} vs {th_aw}'
    )

    # === theta0 unchanged (no mutation contract) ===
    assert t.allclose(theta0, t.tensor([1.0, -1.0])), 'theta0 must not be mutated'

    # === lmda=0 → both paths coincide ===
    th_l2_0, th_aw_0 = ex2_two_step_l2_vs_adamw(theta0, grads, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, lmda=0.0)
    assert t.allclose(th_l2_0, th_aw_0, atol=1e-7), f'lmda=0: paths must coincide, got {th_l2_0} vs {th_aw_0}'

    # === Hand-verify one step of L2 path on a scalar ===
    th = t.tensor([1.0])
    gs = [t.tensor([0.5]), t.tensor([0.5])]
    th_l2_s, _ = ex2_two_step_l2_vs_adamw(th, gs, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, lmda=0.01)
    # Step 1: g' = 0.5 + 0.01*1.0 = 0.51; m=0.051; v=0.0002601
    #         m̂=0.51, v̂=0.2601, step = 0.51/sqrt(0.2601) ≈ 1.0
    #         θ_1 = 1.0 - 0.1*1.0 = 0.9
    # Step 2: g' = 0.5 + 0.01*0.9 = 0.509; carry m, v forward.
    # We don't hand-compute step 2 exactly; just verify it ran (changed).
    assert th_l2_s.item() < 0.9, f'L2 path must take a 2nd step too, got {th_l2_s.item()}'

    # === Both paths apply decay (both move closer to 0 than no-decay would) ===
    th_no_decay, _ = ex2_two_step_l2_vs_adamw(theta0, grads, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, lmda=0.0)
    th_with_decay_l2, th_with_decay_aw = ex2_two_step_l2_vs_adamw(theta0, grads, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, lmda=0.5)
    # Decay should shrink positive components and shrink-magnitude negative components.
    assert th_with_decay_l2[0].abs() < th_no_decay[0].abs() or th_with_decay_aw[0].abs() < th_no_decay[0].abs()

    # === Return type is a 2-tuple of Tensors ===
    ret = ex2_two_step_l2_vs_adamw(theta0, grads, 1e-2, 0.9, 0.999, 1e-8, 0.1)
    assert isinstance(ret, tuple) and len(ret) == 2
    assert isinstance(ret[0], t.Tensor) and isinstance(ret[1], t.Tensor)

    # === Larger lmda → bigger divergence between the two paths ===
    th_l2_a, th_aw_a = ex2_two_step_l2_vs_adamw(theta0, grads, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, lmda=0.05)
    th_l2_b, th_aw_b = ex2_two_step_l2_vs_adamw(theta0, grads, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, lmda=0.5)
    gap_small = (th_l2_a - th_aw_a).abs().max().item()
    gap_large = (th_l2_b - th_aw_b).abs().max().item()
    assert gap_large > gap_small, f'larger lmda must produce larger gap; got {gap_large} not > {gap_small}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_two_step_l2_vs_adamw(theta0, grads, lr, beta1, beta2, eps, lmda):
    assert len(grads) == 2

    # --- L2 path: fold lmda·theta into the gradient stream ---
    theta = theta0.clone()
    m = t.zeros_like(theta)
    v = t.zeros_like(theta)
    for step in (1, 2):
        g = grads[step - 1] + lmda * theta
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        m_hat = m / (1 - beta1 ** step)
        v_hat = v / (1 - beta2 ** step)
        theta = theta - lr * (m_hat / (v_hat.sqrt() + eps))
    theta_l2 = theta

    # --- AdamW path: plain Adam grad, decay applied to theta directly ---
    theta = theta0.clone()
    m = t.zeros_like(theta)
    v = t.zeros_like(theta)
    for step in (1, 2):
        g = grads[step - 1]
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        m_hat = m / (1 - beta1 ** step)
        v_hat = v / (1 - beta2 ** step)
        theta = theta - lr * (m_hat / (v_hat.sqrt() + eps)) - lr * lmda * theta
    theta_adamw = theta

    return theta_l2, theta_adamw
```

**Two-step state matters.** A single Adam step with `m=v=0` is atypical — the bias correction `1 − β^1` exactly cancels the `1 − β` numerator coefficients, making `m̂/√v̂ = sign(g)`. Step 2 is where `m`, `v` actually carry forward and the L2-vs-AdamW gap manifests through the asymmetric `√v̂` normalisation.

**Why bigger `lmda` ⇒ bigger gap.** Both paths apply MORE decay as `lmda` grows, but the L2 path routes that decay through Adam's `/√v̂`, which scales differently per coordinate. Larger `lmda` means larger contribution of `lmda·θ` inside `g`, larger relative asymmetry under normalisation.

**`theta0.clone()` is the no-mutation contract.** Re-using the same starting tensor across both trajectories — without cloning — would link them. The caller passes in `theta0` once; we own the two trajectory copies.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()